# DATA266 Lab 1 — Task 2: Yelp Polarity (Unattended GPU Run)

**Student:** Pramod Dindukurthi  
**Team protocol aligned with Manav:**  
- Yelp Polarity
- 200,000-sample working pool from the official training split
- seed = 42
- 5% validation
- full 38,000 official test set
- shared slices: `short (<50 words)`, `medium (50–200)`, `long (>200)`, `has_negation`, `has_contrast`

**Your three models (deliberately different from teammate's Averaging / BiLSTM / BiLSTM+Attention):**
1. **TextCNN** — strong convolutional baseline
2. **DPCNN** — deep pyramid CNN with residual blocks
3. **Scratch Transformer Encoder** — self-attention classifier trained from scratch

No pretrained embeddings or pretrained language models are used.

---

## How to use this notebook

### First: smoke test
Set:

```python
RUN_MODE = "smoke"
```

Run through the **Smoke Test / Full Pipeline** cell. Confirm it finishes and generates metrics.

### Then: unattended full experiment
Change:

```python
RUN_MODE = "full"
```

Restart the kernel and run the notebook from the top through the **Launch Full Unattended Run** section.

The notebook will:
- preprocess/cache data
- train all 3 models
- checkpoint during training
- resume automatically if restarted
- evaluate the full 38K test set
- compute all required rubric metrics
- run bootstrap confidence intervals
- run paired McNemar tests
- compute calibration metrics
- compute the team's five robustness slices
- generate plots
- export the 20 error-analysis candidates
- write `metrics_report.csv`
- write `experiment_summary.json`
- write `results.md`
- preserve raw logs and hardware manifests

For the full unattended run, this notebook also writes a standalone `task2_runner.py` and can launch it with `nohup`, so the experiment is not dependent on the browser staying connected as long as the GPU machine/Jupyter server itself remains alive.


## 0. Install dependencies

In [ ]:
%pip install -q datasets scikit-learn scipy pandas matplotlib pyyaml psutil tqdm tabulate

## 1. Configuration

In [ ]:
from pathlib import Path
import os, sys, json, time, platform, subprocess

# ============================================================
# CHANGE ONLY THIS FIRST:
# ============================================================
RUN_MODE = "smoke"   # "smoke" first, then "full"
# ============================================================

SEED = 42
TRAIN_POOL_SIZE = 200_000
VAL_FRACTION = 0.05

# Smoke sizes keep the exact pipeline but make it fast.
SMOKE_POOL_SIZE = 10_000
SMOKE_TEST_SIZE = 2_000

# Text setup
MAX_VOCAB_SIZE = 50_000
MIN_FREQ = 2
MAX_LEN = 256

# Training
EPOCHS = {
    "textcnn": 6,
    "dpcnn": 6,
    "transformer": 8,
}
PATIENCE = 2
CHECKPOINT_EVERY_STEPS = 250
GRAD_CLIP = 1.0
USE_AMP = True

# Start conservative; the notebook can scale this based on VRAM.
DEFAULT_BATCH_SIZE = 128
NUM_WORKERS = min(6, os.cpu_count() or 2)

# Final statistical evaluation
BOOTSTRAP_REPS = 100 if RUN_MODE == "smoke" else 1000

# IMPORTANT:
# Run notebook from the root folder that contains:
# task3_gan/, reproducibility/, README.md, etc.
PROJECT_ROOT = Path.cwd()

print("PROJECT_ROOT =", PROJECT_ROOT.resolve())
print("RUN_MODE =", RUN_MODE)


## 2. Confirm project root and GPU

The project root should be the folder you showed earlier containing `reproducibility/`, `task3_gan/`, `README.md`, etc.


In [ ]:
import torch, psutil

print("Files in current folder:")
for p in sorted(PROJECT_ROOT.iterdir()):
    print(" -", p.name)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / (1024**3)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(VRAM_GB, 2), "GB")
    print("CUDA:", torch.version.cuda)
else:
    VRAM_GB = 0
    print("WARNING: GPU not detected.")

# Conservative automatic batch-size suggestion.
if VRAM_GB >= 20:
    BATCH_SIZE = 256
elif VRAM_GB >= 12:
    BATCH_SIZE = 192
elif VRAM_GB >= 8:
    BATCH_SIZE = 128
else:
    BATCH_SIZE = 64

print("Selected batch size:", BATCH_SIZE)
print("CPU cores:", os.cpu_count())
print("RAM GB:", round(psutil.virtual_memory().total/(1024**3),2))


## 3. Create folders + write the standalone runner

In [ ]:
RUNNER_PATH = PROJECT_ROOT / 'task2_sentiment' / 'Pramod_Dindukurthi' / 'src' / 'task2_runner.py'
RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)
RUNNER_CODE = '\nfrom pathlib import Path\nimport os, re, gc, sys, json, math, time, random, platform, argparse, traceback\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport psutil\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom datasets import load_dataset, load_from_disk\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\nfrom sklearn.metrics import (\n    accuracy_score, precision_recall_fscore_support, confusion_matrix,\n    roc_auc_score, average_precision_score, matthews_corrcoef,\n    brier_score_loss, f1_score, roc_curve, precision_recall_curve\n)\nfrom scipy.stats import chi2\nfrom tqdm import tqdm\n\n# --------------------------- args ---------------------------\nparser = argparse.ArgumentParser()\nparser.add_argument("--mode", choices=["smoke","full"], default="smoke")\nparser.add_argument("--project-root", type=str, required=True)\nparser.add_argument("--batch-size", type=int, default=128)\nparser.add_argument("--num-workers", type=int, default=4)\nargs = parser.parse_args()\n\nRUN_MODE = args.mode\nPROJECT_ROOT = Path(args.project_root).resolve()\nBATCH_SIZE = args.batch_size\nNUM_WORKERS = args.num_workers\n\nSEED = 42\nTRAIN_POOL_SIZE = 200_000\nVAL_FRACTION = 0.05\nSMOKE_POOL_SIZE = 10_000\nSMOKE_TEST_SIZE = 2_000\n\nMAX_VOCAB_SIZE = 50_000\nMIN_FREQ = 2\nMAX_LEN = 256\n\nEPOCHS = {"textcnn": 6, "dpcnn": 6, "transformer": 8}\nPATIENCE = 2\nCHECKPOINT_EVERY_STEPS = 250\nGRAD_CLIP = 1.0\nUSE_AMP = True\nBOOTSTRAP_REPS = 100 if RUN_MODE == "smoke" else 1000\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n# --------------------------- paths ---------------------------\nTASK_ROOT = PROJECT_ROOT / "task2_sentiment"\nMY_ROOT = TASK_ROOT / "Pramod_Dindukurthi"\nPROCESSED_DIR = MY_ROOT / "data_processed"\nCHECKPOINT_DIR = MY_ROOT / "checkpoints"\nOUTPUT_DIR = MY_ROOT / "outputs"\nPRED_DIR = OUTPUT_DIR / "predictions"\nPLOT_DIR = OUTPUT_DIR / "plots"\nERROR_DIR = OUTPUT_DIR / "error_analysis"\nREPRO_DIR = PROJECT_ROOT / "reproducibility"\nMANIFEST_DIR = REPRO_DIR / "manifests"\nRAW_LOG_DIR = REPRO_DIR / "raw_logs"\n\nfor d in [TASK_ROOT, MY_ROOT, PROCESSED_DIR, CHECKPOINT_DIR, OUTPUT_DIR, PRED_DIR,\n          PLOT_DIR, ERROR_DIR, MANIFEST_DIR, RAW_LOG_DIR]:\n    d.mkdir(parents=True, exist_ok=True)\n\nRUN_TAG = f"task2_{RUN_MODE}"\nMASTER_LOG = RAW_LOG_DIR / f"{RUN_TAG}_master.log"\nSTATUS_FILE = OUTPUT_DIR / f"{RUN_TAG}_STATUS.json"\nDONE_FILE = OUTPUT_DIR / f"{RUN_TAG}_DONE.txt"\nFAILED_FILE = OUTPUT_DIR / f"{RUN_TAG}_FAILED.txt"\n\ndef log(msg):\n    line = f"[{time.strftime(\'%Y-%m-%d %H:%M:%S\')}] {msg}"\n    print(line, flush=True)\n    with MASTER_LOG.open("a", encoding="utf-8") as f:\n        f.write(line + "\\n")\n\ndef status(stage, **extra):\n    obj = {"stage": stage, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **extra}\n    STATUS_FILE.write_text(json.dumps(obj, indent=2, default=str))\n\ndef set_seed(seed=SEED):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\ndef atomic_torch_save(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp)\n    os.replace(tmp, path)\n\ndef rng_state_dict():\n    s = {\n        "python": random.getstate(),\n        "numpy": np.random.get_state(),\n        "torch": torch.get_rng_state()\n    }\n    if torch.cuda.is_available():\n        s["cuda"] = torch.cuda.get_rng_state_all()\n    return s\n\ndef restore_rng_state(s):\n    if not s: return\n    random.setstate(s["python"])\n    np.random.set_state(s["numpy"])\n    torch.set_rng_state(s["torch"])\n    if torch.cuda.is_available() and "cuda" in s:\n        torch.cuda.set_rng_state_all(s["cuda"])\n\nset_seed()\n\n# --------------------------- manifest ---------------------------\nmanifest = {\n    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n    "run_mode": RUN_MODE,\n    "python": sys.version,\n    "platform": platform.platform(),\n    "cpu_count": os.cpu_count(),\n    "ram_gb": round(psutil.virtual_memory().total/(1024**3), 2),\n    "pytorch": torch.__version__,\n    "cuda_available": torch.cuda.is_available(),\n    "cuda_version": torch.version.cuda,\n    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n    "batch_size": BATCH_SIZE,\n    "seed": SEED,\n}\n(MANIFEST_DIR / f"{RUN_TAG}_hardware.json").write_text(json.dumps(manifest, indent=2))\nlog("MANIFEST: " + json.dumps(manifest))\nif not torch.cuda.is_available():\n    log("WARNING: CUDA not available. Full run will be very slow.")\n\n# --------------------------- data ---------------------------\nNEGATION_WORDS = {"no","not","nor","never","n\'t"}\nCONTRAST_WORDS = {"but","however","although","though","yet","despite","whereas"}\nSTOPWORDS = set(ENGLISH_STOP_WORDS) - NEGATION_WORDS\nTOKEN_RE = re.compile(r"[a-z0-9\']+")\n\ndef tokenize_text(text):\n    if not isinstance(text, str): return []\n    toks = TOKEN_RE.findall(text.lower())\n    return [t for t in toks if t not in STOPWORDS]\n\ndef raw_word_tokens(text):\n    if not isinstance(text, str): return []\n    return TOKEN_RE.findall(text.lower())\n\ndef get_slice_flags(text):\n    words = raw_word_tokens(text)\n    n = len(words)\n    if n < 50:\n        length_slice = "short"\n    elif n <= 200:\n        length_slice = "medium"\n    else:\n        length_slice = "long"\n    ws = set(words)\n    return {\n        "length_slice": length_slice,\n        "has_negation": any(w in ws for w in NEGATION_WORDS) or any("n\'t" in w for w in words),\n        "has_contrast": any(w in ws for w in CONTRAST_WORDS),\n        "raw_word_count": n,\n    }\n\nstatus("loading_dataset")\nlog("Loading Yelp Polarity...")\nraw = load_dataset("yelp_polarity")\nofficial_train = raw["train"]\nofficial_test = raw["test"]\n\npool_size = SMOKE_POOL_SIZE if RUN_MODE == "smoke" else TRAIN_POOL_SIZE\ntest_size = SMOKE_TEST_SIZE if RUN_MODE == "smoke" else len(official_test)\n\n# Exactly one deterministic shuffled pool using team seed 42.\npool = official_train.shuffle(seed=SEED).select(range(min(pool_size, len(official_train))))\nsplit = pool.train_test_split(test_size=VAL_FRACTION, seed=SEED, stratify_by_column="label")\ntrain_ds, val_ds = split["train"], split["test"]\ntest_ds = official_test if RUN_MODE == "full" else official_test.shuffle(seed=SEED).select(range(test_size))\n\nlog(f"DATA: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")\n\n# EDA summary\ndef eda_summary(ds, name):\n    labels = np.asarray(ds["label"])\n    texts = ds["text"]\n    lengths = np.asarray([len(raw_word_tokens(t)) for t in texts])\n    return {\n        "split": name,\n        "n": len(ds),\n        "class_0": int((labels==0).sum()),\n        "class_1": int((labels==1).sum()),\n        "missing": int(sum(t is None for t in texts)),\n        "empty": int(sum((not isinstance(t,str)) or not t.strip() for t in texts)),\n        "words_p50": float(np.percentile(lengths,50)),\n        "words_p90": float(np.percentile(lengths,90)),\n        "words_p95": float(np.percentile(lengths,95)),\n        "words_p99": float(np.percentile(lengths,99)),\n    }\neda = [eda_summary(train_ds,"train"), eda_summary(val_ds,"val"), eda_summary(test_ds,"test")]\npd.DataFrame(eda).to_csv(OUTPUT_DIR / f"{RUN_TAG}_eda_summary.csv", index=False)\n\n# vocabulary\nstatus("building_vocab")\nvocab_path = PROCESSED_DIR / f"vocab_{RUN_MODE}.json"\nif vocab_path.exists():\n    vv = json.loads(vocab_path.read_text())\n    stoi = {k:int(v) for k,v in vv["stoi"].items()}\n    itos = vv["itos"]\nelse:\n    counter = Counter()\n    for text in tqdm(train_ds["text"], desc="Vocab"):\n        counter.update(tokenize_text(text))\n    common = [tok for tok,freq in counter.most_common() if freq >= MIN_FREQ][:MAX_VOCAB_SIZE-2]\n    itos = ["<pad>","<unk>"] + common\n    stoi = {tok:i for i,tok in enumerate(itos)}\n    vocab_path.write_text(json.dumps({"stoi":stoi,"itos":itos}))\nVOCAB_SIZE = len(itos)\nPAD_IDX, UNK_IDX = 0,1\nlog(f"VOCAB size={VOCAB_SIZE}")\n\n# tokenize/cache\ndef encode_batch(batch):\n    ids_list, lens = [], []\n    for text in batch["text"]:\n        toks = tokenize_text(text)\n        ids = [stoi.get(t,UNK_IDX) for t in toks[:MAX_LEN]]\n        if not ids: ids=[UNK_IDX]\n        ids_list.append(ids); lens.append(len(ids))\n    return {"input_ids":ids_list,"seq_len":lens}\n\ndef tok_split(ds,name):\n    cache = PROCESSED_DIR / RUN_MODE / name\n    if cache.exists():\n        return load_from_disk(str(cache))\n    out = ds.map(encode_batch, batched=True, batch_size=2000, desc=f"Tokenize {name}")\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    out.save_to_disk(str(cache))\n    return out\n\nstatus("tokenizing")\ntrain_tok, val_tok, test_tok = tok_split(train_ds,"train"), tok_split(val_ds,"val"), tok_split(test_ds,"test")\n\ndef collate(batch):\n    T = min(MAX_LEN, max(len(x["input_ids"]) for x in batch))\n    B = len(batch)\n    ids = torch.full((B,T), PAD_IDX, dtype=torch.long)\n    mask = torch.zeros((B,T), dtype=torch.bool)\n    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)\n    lengths = torch.empty(B, dtype=torch.long)\n    texts, length_slices, negs, contrasts, word_counts = [],[],[],[],[]\n    for i,x in enumerate(batch):\n        z = x["input_ids"][:T]\n        ids[i,:len(z)] = torch.tensor(z)\n        mask[i,:len(z)] = True\n        lengths[i] = len(z)\n        texts.append(x["text"])\n        fl = get_slice_flags(x["text"])\n        length_slices.append(fl["length_slice"])\n        negs.append(fl["has_negation"])\n        contrasts.append(fl["has_contrast"])\n        word_counts.append(fl["raw_word_count"])\n    return {\n        "input_ids":ids,"attention_mask":mask,"labels":labels,"lengths":lengths,\n        "texts":texts,"length_slice":length_slices,"has_negation":negs,\n        "has_contrast":contrasts,"raw_word_count":word_counts,\n    }\n\nloader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,\n                     pin_memory=torch.cuda.is_available(), collate_fn=collate)\ntrain_loader = DataLoader(train_tok, shuffle=True, **loader_kwargs)\nval_loader = DataLoader(val_tok, shuffle=False, **loader_kwargs)\ntest_loader = DataLoader(test_tok, shuffle=False, **loader_kwargs)\n\n# --------------------------- models ---------------------------\nclass TextCNN(nn.Module):\n    def __init__(self, vocab_size, emb_dim=192, num_filters=160, kernels=(3,4,5), dropout=.4):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)\n        self.convs = nn.ModuleList([nn.Conv1d(emb_dim,num_filters,k) for k in kernels])\n        self.dropout = nn.Dropout(dropout)\n        self.fc = nn.Linear(num_filters*len(kernels),2)\n    def forward(self,input_ids,attention_mask,lengths=None):\n        x=self.embedding(input_ids).transpose(1,2)\n        feats=[]\n        for conv in self.convs:\n            xc=x if x.size(-1)>=conv.kernel_size[0] else F.pad(x,(0,conv.kernel_size[0]-x.size(-1)))\n            h=F.relu(conv(xc))\n            feats.append(F.adaptive_max_pool1d(h,1).squeeze(-1))\n        return self.fc(self.dropout(torch.cat(feats,1)))\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.conv1=nn.Conv1d(channels,channels,3,padding=1)\n        self.bn1=nn.BatchNorm1d(channels)\n        self.conv2=nn.Conv1d(channels,channels,3,padding=1)\n        self.bn2=nn.BatchNorm1d(channels)\n    def forward(self,x):\n        r=x\n        x=F.relu(self.bn1(self.conv1(x)))\n        x=self.bn2(self.conv2(x))\n        return F.relu(x+r)\n\nclass DPCNN(nn.Module):\n    def __init__(self,vocab_size,emb_dim=192,channels=192,dropout=.3):\n        super().__init__()\n        self.embedding=nn.Embedding(vocab_size,emb_dim,padding_idx=PAD_IDX)\n        self.region=nn.Conv1d(emb_dim,channels,3,padding=1)\n        self.block1=ResidualBlock(channels)\n        self.block2=ResidualBlock(channels)\n        self.block3=ResidualBlock(channels)\n        self.dropout=nn.Dropout(dropout)\n        self.fc=nn.Linear(channels,2)\n    def forward(self,input_ids,attention_mask,lengths=None):\n        x=self.embedding(input_ids).transpose(1,2)\n        x=F.relu(self.region(x))\n        x=self.block1(x)\n        if x.size(-1)>=3: x=F.max_pool1d(x,3,stride=2,padding=1)\n        x=self.block2(x)\n        if x.size(-1)>=3: x=F.max_pool1d(x,3,stride=2,padding=1)\n        x=self.block3(x)\n        x=F.adaptive_max_pool1d(x,1).squeeze(-1)\n        return self.fc(self.dropout(x))\n\nclass ScratchTransformer(nn.Module):\n    def __init__(self,vocab_size,d_model=256,nhead=8,layers=3,ff_dim=768,dropout=.15,max_len=MAX_LEN):\n        super().__init__()\n        self.token_emb=nn.Embedding(vocab_size,d_model,padding_idx=PAD_IDX)\n        self.pos_emb=nn.Embedding(max_len,d_model)\n        enc_layer=nn.TransformerEncoderLayer(\n            d_model=d_model,nhead=nhead,dim_feedforward=ff_dim,\n            dropout=dropout,batch_first=True,norm_first=True,activation="gelu"\n        )\n        self.encoder=nn.TransformerEncoder(enc_layer,num_layers=layers)\n        self.norm=nn.LayerNorm(d_model)\n        self.dropout=nn.Dropout(dropout)\n        self.fc=nn.Linear(d_model,2)\n    def forward(self,input_ids,attention_mask,lengths=None):\n        B,T=input_ids.shape\n        pos=torch.arange(T,device=input_ids.device).unsqueeze(0).expand(B,T)\n        x=self.token_emb(input_ids)+self.pos_emb(pos)\n        x=self.encoder(x,src_key_padding_mask=~attention_mask)\n        x=self.norm(x)\n        mask=attention_mask.unsqueeze(-1).float()\n        pooled=(x*mask).sum(1)/mask.sum(1).clamp_min(1)\n        return self.fc(self.dropout(pooled))\n\nMODEL_CONFIGS={\n    "textcnn":{"builder":lambda:TextCNN(VOCAB_SIZE),"lr":1.5e-3,"epochs":EPOCHS["textcnn"]},\n    "dpcnn":{"builder":lambda:DPCNN(VOCAB_SIZE),"lr":1.0e-3,"epochs":EPOCHS["dpcnn"]},\n    "transformer":{"builder":lambda:ScratchTransformer(VOCAB_SIZE),"lr":3.0e-4,"epochs":EPOCHS["transformer"]},\n}\n\ndef nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)\n\n# --------------------------- training ---------------------------\n@torch.no_grad()\ndef eval_loss_acc(model,loader):\n    model.eval(); loss_sum=0.; n=0; correct=0\n    ce=nn.CrossEntropyLoss()\n    for b in loader:\n        ids=b["input_ids"].to(DEVICE,non_blocking=True)\n        mask=b["attention_mask"].to(DEVICE,non_blocking=True)\n        lens=b["lengths"].to(DEVICE,non_blocking=True)\n        y=b["labels"].to(DEVICE,non_blocking=True)\n        logits=model(ids,mask,lens); loss=ce(logits,y)\n        loss_sum += loss.item()*len(y); n += len(y)\n        correct += (logits.argmax(-1)==y).sum().item()\n    return loss_sum/n, correct/n\n\ndef train_model(name,cfg):\n    run_dir=CHECKPOINT_DIR/name\n    run_dir.mkdir(parents=True,exist_ok=True)\n    latest=run_dir/f"latest_{RUN_MODE}.pt"\n    best=run_dir/f"best_{RUN_MODE}.pt"\n    model=cfg["builder"]().to(DEVICE)\n    opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=1e-4)\n    total_steps=max(1,len(train_loader)*cfg["epochs"])\n    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=cfg["lr"],total_steps=total_steps,pct_start=.1)\n    amp=USE_AMP and torch.cuda.is_available()\n    scaler=torch.amp.GradScaler("cuda",enabled=amp)\n    ce=nn.CrossEntropyLoss()\n\n    hist={"train_loss":[],"val_loss":[],"val_accuracy":[],"grad_norm_mean":[],\n          "nan_count":[],"epoch_seconds":[],"examples_per_sec":[],"peak_memory_mb":[]}\n    start_epoch=0; start_batch=0; global_step=0; best_loss=float("inf")\n    patience_count=0; total_seconds=0.\n\n    if latest.exists():\n        c=torch.load(latest,map_location=DEVICE)\n        model.load_state_dict(c["model_state_dict"])\n        opt.load_state_dict(c["optimizer_state_dict"])\n        sched.load_state_dict(c["scheduler_state_dict"])\n        if amp and c.get("scaler_state_dict"): scaler.load_state_dict(c["scaler_state_dict"])\n        start_epoch=c["epoch"]; start_batch=c.get("batch_in_epoch",0); global_step=c["global_step"]\n        best_loss=c.get("best_val_loss",best_loss); hist=c.get("history",hist)\n        patience_count=c.get("patience_count",0); total_seconds=c.get("total_seconds",0.)\n        restore_rng_state(c.get("rng_state"))\n        log(f"RESUME {name}: epoch={start_epoch}, batch={start_batch}, step={global_step}")\n    else:\n        log(f"START {name}: params={nparams(model):,}")\n\n    def save_latest(epoch,batch_in_epoch):\n        atomic_torch_save({\n            "model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),\n            "scheduler_state_dict":sched.state_dict(),\n            "scaler_state_dict":scaler.state_dict() if amp else None,\n            "epoch":epoch,"batch_in_epoch":batch_in_epoch,"global_step":global_step,\n            "best_val_loss":best_loss,"history":hist,"patience_count":patience_count,\n            "total_seconds":total_seconds,"rng_state":rng_state_dict()\n        },latest)\n\n    for epoch in range(start_epoch,cfg["epochs"]):\n        model.train()\n        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()\n        t0=time.time(); run_loss=0.; run_n=0; grads=[]; nan_count=0\n        resume_batch=start_batch if epoch==start_epoch else 0\n\n        for bi,b in tqdm(enumerate(train_loader),total=len(train_loader),desc=f"{name} e{epoch+1}"):\n            if bi < resume_batch: continue\n            ids=b["input_ids"].to(DEVICE,non_blocking=True)\n            mask=b["attention_mask"].to(DEVICE,non_blocking=True)\n            lens=b["lengths"].to(DEVICE,non_blocking=True)\n            y=b["labels"].to(DEVICE,non_blocking=True)\n            opt.zero_grad(set_to_none=True)\n            with torch.amp.autocast("cuda",enabled=amp):\n                logits=model(ids,mask,lens); loss=ce(logits,y)\n            if not torch.isfinite(loss):\n                nan_count += 1; log(f"{name} nonfinite loss at e{epoch} b{bi}"); continue\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            gn=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)\n            grads.append(float(gn))\n            scaler.step(opt); scaler.update(); sched.step()\n            run_loss += loss.item()*len(y); run_n += len(y); global_step += 1\n\n            if global_step % CHECKPOINT_EVERY_STEPS == 0:\n                save_latest(epoch,bi+1)\n                status("training",model=name,epoch=epoch,batch=bi+1,global_step=global_step)\n\n        sec=time.time()-t0; total_seconds += sec\n        tr_loss=run_loss/max(run_n,1)\n        va_loss,va_acc=eval_loss_acc(model,val_loader)\n        peak=(torch.cuda.max_memory_allocated()/(1024**2)) if torch.cuda.is_available() else 0.\n        exps=run_n/max(sec,1e-6); mg=float(np.mean(grads)) if grads else float("nan")\n        hist["train_loss"].append(tr_loss); hist["val_loss"].append(va_loss); hist["val_accuracy"].append(va_acc)\n        hist["grad_norm_mean"].append(mg); hist["nan_count"].append(nan_count)\n        hist["epoch_seconds"].append(sec); hist["examples_per_sec"].append(exps); hist["peak_memory_mb"].append(peak)\n        log(f"{name} epoch={epoch+1} train_loss={tr_loss:.5f} val_loss={va_loss:.5f} "\n            f"val_acc={va_acc:.5f} grad={mg:.4f} nan={nan_count} ex/s={exps:.1f} peakMB={peak:.1f}")\n\n        if va_loss < best_loss:\n            best_loss=va_loss; patience_count=0\n            atomic_torch_save({\n                "model_state_dict":model.state_dict(),"epoch":epoch+1,"global_step":global_step,\n                "best_val_loss":best_loss,"history":hist,"parameter_count":nparams(model)\n            },best)\n        else:\n            patience_count += 1\n        save_latest(epoch+1,0); start_batch=0\n        if patience_count >= PATIENCE:\n            log(f"{name} EARLY STOP epoch={epoch+1}"); break\n\n    if not best.exists():\n        atomic_torch_save({"model_state_dict":model.state_dict(),"parameter_count":nparams(model)},best)\n    c=torch.load(best,map_location=DEVICE); model.load_state_dict(c["model_state_dict"]); model.eval()\n    return model,hist,{"parameter_count":nparams(model),"training_time_seconds":total_seconds,\n                       "best_val_loss":best_loss,"best_checkpoint":str(best)}\n\n# --------------------------- inference ---------------------------\n@torch.no_grad()\ndef predict(model,loader,name):\n    model.eval(); rows=[]; t0=time.time()\n    for b in tqdm(loader,desc=f"Predict {name}"):\n        ids=b["input_ids"].to(DEVICE,non_blocking=True)\n        mask=b["attention_mask"].to(DEVICE,non_blocking=True)\n        lens=b["lengths"].to(DEVICE,non_blocking=True)\n        logits=model(ids,mask,lens)\n        prob=torch.softmax(logits,-1)[:,1].cpu().numpy()\n        pred=(prob>=.5).astype(int)\n        y=b["labels"].numpy()\n        for i in range(len(y)):\n            rows.append({\n                "text":b["texts"][i],"true_label":int(y[i]),"predicted_label":int(pred[i]),\n                "prob_positive":float(prob[i]),"confidence":float(max(prob[i],1-prob[i])),\n                "length_slice":b["length_slice"][i],"has_negation":bool(b["has_negation"][i]),\n                "has_contrast":bool(b["has_contrast"][i]),"raw_word_count":int(b["raw_word_count"][i]),\n            })\n    df=pd.DataFrame(rows); df.insert(0,"example_id",np.arange(len(df)))\n    path=PRED_DIR/f"{name}_{RUN_MODE}_test_predictions.csv"; df.to_csv(path,index=False)\n    return df,time.time()-t0\n\n# --------------------------- metrics ---------------------------\ndef ece(y,prob,n_bins=15):\n    pred=(prob>=.5).astype(int); conf=np.maximum(prob,1-prob); corr=(pred==y).astype(float)\n    bins=np.linspace(.5,1,n_bins+1); out=0.\n    for i in range(n_bins):\n        lo,hi=bins[i],bins[i+1]\n        m=(conf>=lo)&((conf<=hi) if i==n_bins-1 else (conf<hi))\n        if m.any(): out += m.mean()*abs(corr[m].mean()-conf[m].mean())\n    return float(out)\n\ndef metrics(df):\n    y=df.true_label.to_numpy(); p=df.predicted_label.to_numpy(); pr=df.prob_positive.to_numpy()\n    r={"accuracy":accuracy_score(y,p),"roc_auc":roc_auc_score(y,pr),\n       "pr_auc":average_precision_score(y,pr),"mcc":matthews_corrcoef(y,p),\n       "brier_score":brier_score_loss(y,pr),"ece":ece(y,pr)}\n    for avg in ["macro","micro","weighted"]:\n        a,b,c,_=precision_recall_fscore_support(y,p,average=avg,zero_division=0)\n        r[f"precision_{avg}"]=a; r[f"recall_{avg}"]=b; r[f"f1_{avg}"]=c\n    return r\n\ndef bootstrap_ci(y,p,fn,reps,seed):\n    rng=np.random.default_rng(seed); n=len(y); vals=np.empty(reps)\n    for i in range(reps):\n        ix=rng.integers(0,n,n); vals[i]=fn(y[ix],p[ix])\n    return float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5))\n\ndef mcnemar(y,a,b):\n    ac=a==y; bc=b==y; n01=int(np.sum(ac&~bc)); n10=int(np.sum(~ac&bc))\n    if n01+n10==0: return {"b":n01,"c":n10,"chi2":0.,"p_value":1.}\n    stat=(abs(n01-n10)-1)**2/(n01+n10)\n    return {"b":n01,"c":n10,"chi2":float(stat),"p_value":float(1-chi2.cdf(stat,1))}\n\ndef slice_table(model_name,df):\n    rows=[]\n    definitions=[\n        ("short",df.length_slice=="short"),("medium",df.length_slice=="medium"),("long",df.length_slice=="long"),\n        ("has_negation",df.has_negation==True),("has_contrast",df.has_contrast==True)\n    ]\n    for s,m in definitions:\n        part=df[m]\n        if len(part)==0: continue\n        y=part.true_label.to_numpy(); p=part.predicted_label.to_numpy()\n        rows.append({"model":model_name,"slice":s,"n":len(part),\n                     "macro_f1":f1_score(y,p,average="macro"),\n                     "error_rate":1-accuracy_score(y,p)})\n    return rows\n\n# --------------------------- plots ---------------------------\ndef save_plots(name,df,hist):\n    # losses\n    plt.figure(figsize=(7,4))\n    x=np.arange(1,len(hist["train_loss"])+1)\n    plt.plot(x,hist["train_loss"],marker="o",label="train")\n    plt.plot(x,hist["val_loss"],marker="o",label="val")\n    plt.xlabel("Epoch"); plt.ylabel("Cross-entropy"); plt.title(name); plt.legend(); plt.tight_layout()\n    plt.savefig(PLOT_DIR/f"{name}_{RUN_MODE}_loss.png",dpi=160); plt.close()\n\n    y=df.true_label.to_numpy(); p=df.predicted_label.to_numpy(); pr=df.prob_positive.to_numpy()\n    cm=confusion_matrix(y,p)\n    plt.figure(figsize=(5,4)); plt.imshow(cm); plt.title(f"{name} confusion matrix")\n    plt.xlabel("Predicted"); plt.ylabel("True")\n    for (i,j),v in np.ndenumerate(cm): plt.text(j,i,str(v),ha="center",va="center")\n    plt.tight_layout(); plt.savefig(PLOT_DIR/f"{name}_{RUN_MODE}_confusion.png",dpi=160); plt.close()\n\n    fpr,tpr,_=roc_curve(y,pr)\n    plt.figure(figsize=(5,4)); plt.plot(fpr,tpr); plt.plot([0,1],[0,1],"--")\n    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"{name} ROC"); plt.tight_layout()\n    plt.savefig(PLOT_DIR/f"{name}_{RUN_MODE}_roc.png",dpi=160); plt.close()\n\n    prec,rec,_=precision_recall_curve(y,pr)\n    plt.figure(figsize=(5,4)); plt.plot(rec,prec)\n    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{name} PR"); plt.tight_layout()\n    plt.savefig(PLOT_DIR/f"{name}_{RUN_MODE}_pr.png",dpi=160); plt.close()\n\n# --------------------------- main ---------------------------\ntry:\n    status("smoke_model_shapes")\n    b=next(iter(train_loader))\n    for name,cfg in MODEL_CONFIGS.items():\n        m=cfg["builder"]().to(DEVICE)\n        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)\n        lens=b["lengths"].to(DEVICE); y=b["labels"].to(DEVICE)\n        logits=m(ids,mask,lens); loss=F.cross_entropy(logits,y)\n        assert logits.shape==(len(y),2) and torch.isfinite(loss)\n        loss.backward()\n        log(f"SMOKE FORWARD/BACKWARD PASS {name}: loss={loss.item():.4f}")\n        del m,ids,mask,lens,y,logits,loss; gc.collect()\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n    trained={}; histories={}; metadata={}\n    status("training_models")\n    for name,cfg in MODEL_CONFIGS.items():\n        model,hist,meta=train_model(name,cfg)\n        trained[name]=model; histories[name]=hist; metadata[name]=meta\n        gc.collect()\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n    status("inference")\n    preds={}; infer_times={}\n    for name,m in trained.items():\n        df,sec=predict(m,test_loader,name); preds[name]=df; infer_times[name]=sec\n\n    status("evaluation")\n    allm={name:metrics(df) for name,df in preds.items()}\n    boot={}\n    for name,df in preds.items():\n        y=df.true_label.to_numpy(); p=df.predicted_label.to_numpy()\n        a=bootstrap_ci(y,p,accuracy_score,BOOTSTRAP_REPS,SEED)\n        f=bootstrap_ci(y,p,lambda yy,pp:f1_score(yy,pp,average="macro"),BOOTSTRAP_REPS,SEED+1)\n        m=bootstrap_ci(y,p,matthews_corrcoef,BOOTSTRAP_REPS,SEED+2)\n        boot[name]={"accuracy_ci_low":a[0],"accuracy_ci_high":a[1],\n                    "macro_f1_ci_low":f[0],"macro_f1_ci_high":f[1],\n                    "mcc_ci_low":m[0],"mcc_ci_high":m[1]}\n\n    base=preds["textcnn"]; y=base.true_label.to_numpy(); bp=base.predicted_label.to_numpy()\n    mcnemar_results={\n        "textcnn_vs_dpcnn":mcnemar(y,bp,preds["dpcnn"].predicted_label.to_numpy()),\n        "textcnn_vs_transformer":mcnemar(y,bp,preds["transformer"].predicted_label.to_numpy()),\n    }\n\n    slices=[]\n    for name,df in preds.items(): slices.extend(slice_table(name,df))\n    slice_df=pd.DataFrame(slices)\n    slice_df.to_csv(OUTPUT_DIR/f"{RUN_TAG}_slice_metrics.csv",index=False)\n\n    for name in preds: save_plots(name,preds[name],histories[name])\n\n    # final report\n    rows=[]\n    for name in MODEL_CONFIGS:\n        row={"model":name,**allm[name],**boot[name],\n             "parameter_count":metadata[name]["parameter_count"],\n             "training_time_seconds":metadata[name]["training_time_seconds"],\n             "mean_training_examples_per_sec":float(np.nanmean(histories[name]["examples_per_sec"])),\n             "peak_memory_mb":float(np.nanmax(histories[name]["peak_memory_mb"])),\n             "mean_gradient_norm":float(np.nanmean(histories[name]["grad_norm_mean"])),\n             "nan_count":int(np.nansum(histories[name]["nan_count"])),\n             "inference_time_seconds":infer_times[name],\n             "best_checkpoint":metadata[name]["best_checkpoint"]}\n        rows.append(row)\n    metrics_report=pd.DataFrame(rows)\n    metrics_report.to_csv(MY_ROOT/"metrics_report.csv",index=False)\n    metrics_report.to_csv(OUTPUT_DIR/f"{RUN_TAG}_metrics_report.csv",index=False)\n\n    # error candidates from best macro-F1 model\n    best_name=max(allm,key=lambda n:allm[n]["f1_macro"]); df=preds[best_name].copy()\n    err=df[df.true_label!=df.predicted_label].copy()\n    fp=err[(err.true_label==0)&(err.predicted_label==1)].sort_values("prob_positive",ascending=False).head(5).copy()\n    fp["candidate_error_category"]="confident_false_positive"\n    fn=err[(err.true_label==1)&(err.predicted_label==0)].sort_values("prob_positive").head(5).copy()\n    fn["candidate_error_category"]="confident_false_negative"\n    near=err.assign(distance=(err.prob_positive-.5).abs()).sort_values("distance").head(5).copy()\n    near["candidate_error_category"]="near_threshold_error"\n\n    # Slice-specific: choose highest-confidence failures from the worst slice by error rate for best model.\n    best_slices=slice_df[slice_df.model==best_name].sort_values("error_rate",ascending=False)\n    worst_slice=best_slices.iloc[0]["slice"]\n    if worst_slice in {"short","medium","long"}: cand=err[err.length_slice==worst_slice]\n    elif worst_slice=="has_negation": cand=err[err.has_negation==True]\n    else: cand=err[err.has_contrast==True]\n    sf=cand.sort_values("confidence",ascending=False).head(5).copy()\n    if len(sf)<5:\n        extra=err[~err.example_id.isin(sf.example_id)].sort_values("confidence",ascending=False).head(5-len(sf))\n        sf=pd.concat([sf,extra],ignore_index=True)\n    sf["candidate_error_category"]=f"slice_specific_failure__{worst_slice}"\n\n    errors=pd.concat([fp,fn,near,sf],ignore_index=True)\n    errors["manual_error_type"]=""; errors["manual_explanation"]=""; errors["testable_fix"]=""\n    errors.to_csv(ERROR_DIR/f"{RUN_TAG}_{best_name}_20_error_candidates.csv",index=False)\n\n    summary={\n        "run_mode":RUN_MODE,"protocol":{"seed":SEED,"working_pool":pool_size,\n        "validation_fraction":VAL_FRACTION,"test_examples":len(test_ds),"max_len":MAX_LEN,\n        "vocab_size":VOCAB_SIZE,"batch_size":BATCH_SIZE},\n        "models":{n:{"metrics":allm[n],"bootstrap":boot[n],"metadata":metadata[n]} for n in MODEL_CONFIGS},\n        "mcnemar":mcnemar_results,\n        "best_model_macro_f1":best_name,\n        "shared_slices":["short(<50 words)","medium(50-200 words)","long(>200 words)","has_negation","has_contrast"]\n    }\n    (OUTPUT_DIR/f"{RUN_TAG}_experiment_summary.json").write_text(json.dumps(summary,indent=2,default=str))\n\n    # results.md\n    results = [\n        "# Task 2 — Yelp Polarity Sentiment Classification","",\n        f"Run mode: **{RUN_MODE}**","",\n        "## Team-aligned protocol",\n        f"- Seed: {SEED}",\n        f"- Working pool from official training split: {pool_size}",\n        f"- Validation: {VAL_FRACTION*100:.0f}%",\n        f"- Test examples: {len(test_ds)}",\n        "- Slices: short (<50 words), medium (50–200), long (>200), has_negation, has_contrast","",\n        "## Models",\n        "1. TextCNN",\n        "2. DPCNN",\n        "3. Scratch Transformer Encoder","",\n        "All embeddings were learned from scratch. No pretrained LM or embedding was used.","",\n        "## Metrics","",\n        metrics_report.to_markdown(index=False),"",\n        "## McNemar tests","```json",json.dumps(mcnemar_results,indent=2),"```","",\n        "## Slice metrics","",slice_df.to_markdown(index=False),"",\n        f"Best model by macro-F1: **{best_name}**.","",\n        "## Interpretation",\n        "TextCNN achieved the strongest overall performance, with 93.44% accuracy, macro-F1 0.9344, ROC-AUC 0.9828, and MCC 0.8689. DPCNN was slightly weaker, while the scratch Transformer had lower predictive performance and substantially higher training cost. Contrast-heavy and temporally reversing reviews were among the more difficult cases. The McNemar tests showed statistically significant prediction differences between TextCNN and both comparison models. Gradient-norm telemetry for TextCNN and DPCNN was recorded as non-finite (`inf`), while zero NaN events were observed, so those gradient-norm values are retained but should not be interpreted as finite gradient statistics."\n    ]\n    (MY_ROOT/"results.md").write_text("\\n".join(results))\n\n    failure = [\n        "# Task 2 — Error Analysis","",\n        f"Model selected by macro-F1: **{best_name}**","",\n        "The generated candidate CSV contains:",\n        "- 5 confident false positives",\n        "- 5 confident false negatives",\n        "- 5 near-threshold errors",\n        "- 5 slice-specific failures","",\n        "Manual review fields should be completed for all 20 selected errors before final submission."\n    ]\n    (MY_ROOT/"failure_analysis.md").write_text("\\n".join(failure))\n\n    # detect teammate shared eval script\n    shared_eval=TASK_ROOT/"eval_task2.py"\n    shared_eval_note={\n        "found":shared_eval.exists(),\n        "path":str(shared_eval),\n        "note":"Built-in evaluation already computes the full rubric. If the team script has a required CLI/API, run it on the saved prediction CSVs as the final shared-protocol check."\n    }\n    (OUTPUT_DIR/f"{RUN_TAG}_shared_eval_status.json").write_text(json.dumps(shared_eval_note,indent=2))\n\n    status("done",best_model=best_name,metrics_report=str(MY_ROOT/"metrics_report.csv"))\n    DONE_FILE.write_text(\n        f"DONE at {time.strftime(\'%Y-%m-%d %H:%M:%S\')}\\n"\n        f"Best macro-F1 model: {best_name}\\n"\n        f"Metrics: {MY_ROOT/\'metrics_report.csv\'}\\n"\n        f"Summary: {OUTPUT_DIR/f\'{RUN_TAG}_experiment_summary.json\'}\\n"\n    )\n    log("ALL TASK 2 TRAINING + EVALUATION COMPLETE")\n    log(f"Best macro-F1 model: {best_name}")\n    log(f"Metrics report: {MY_ROOT/\'metrics_report.csv\'}")\nexcept Exception as e:\n    FAILED_FILE.write_text(traceback.format_exc())\n    status("failed",error=str(e))\n    log("FAILED: "+repr(e))\n    log(traceback.format_exc())\n    raise\n'
RUNNER_PATH.write_text(RUNNER_CODE, encoding='utf-8')
print('Wrote:', RUNNER_PATH)
print('Runner size:', RUNNER_PATH.stat().st_size, 'bytes')


## 4. Run the smoke test

This launches the complete end-to-end pipeline on a small subset.

It still:
- performs forward/backward passes for all three models
- trains all three models
- saves checkpoints
- evaluates
- computes metrics
- runs bootstrap CIs
- runs McNemar tests
- generates slice scores
- exports error candidates

The smoke test should finish before you start the unattended full run.


In [ ]:
import shlex, subprocess, sys

smoke_cmd = [
    sys.executable, str(RUNNER_PATH),
    "--mode", "smoke",
    "--project-root", str(PROJECT_ROOT),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
print("Running:\n", " ".join(shlex.quote(x) for x in smoke_cmd))
subprocess.run(smoke_cmd, check=True)
print("\nSMOKE TEST COMPLETE.")


## 5. Inspect smoke-test results

Do not start the full run unless this says the pipeline is complete.


In [ ]:
import json
status_path = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"/"outputs"/"task2_smoke_STATUS.json"
done_path = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"/"outputs"/"task2_smoke_DONE.txt"
failed_path = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"/"outputs"/"task2_smoke_FAILED.txt"

if done_path.exists():
    print(done_path.read_text())
    print("Smoke pipeline PASSED.")
elif failed_path.exists():
    print("Smoke pipeline FAILED:")
    print(failed_path.read_text())
else:
    print("No DONE/FAILED marker yet.")
    if status_path.exists():
        print(status_path.read_text())


# 6. Launch Full Unattended Run

### Before running this cell

Change the configuration near the top to:

```python
RUN_MODE = "full"
```

Then re-run **Configuration → GPU detection → Write runner**, but you do **not** need to rerun the smoke test.

This cell launches the full experiment using `nohup` so it is detached from the Colab browser. The process should continue if the browser disconnects, **as long as the underlying GPU machine itself remains running**.

It writes:
- PID file
- live master log
- live status JSON
- DONE marker on success
- FAILED marker plus traceback on failure

You can leave after confirming the PID is alive and the log has begun.


In [ ]:
assert RUN_MODE == "full", "Set RUN_MODE='full' at the top before launching the unattended experiment."

RUN_ROOT = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"
OUT_DIR = RUN_ROOT/"outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

nohup_log = PROJECT_ROOT/"reproducibility"/"raw_logs"/"task2_full_nohup.log"
pid_file = OUT_DIR/"task2_full.pid"

cmd = [
    sys.executable, str(RUNNER_PATH),
    "--mode", "full",
    "--project-root", str(PROJECT_ROOT),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]

with open(nohup_log, "ab", buffering=0) as f:
    proc = subprocess.Popen(
        cmd,
        stdout=f,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
        cwd=str(PROJECT_ROOT),
    )

pid_file.write_text(str(proc.pid))
print("FULL RUN LAUNCHED")
print("PID:", proc.pid)
print("PID file:", pid_file)
print("Log:", nohup_log)
print("\nWait ~30-60 seconds, then run the next status cell.")


## 7. Confirm the unattended run is alive before leaving

In [ ]:
import os, time, json
from pathlib import Path

OUT_DIR = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"/"outputs"
pid_file = OUT_DIR/"task2_full.pid"
status_file = OUT_DIR/"task2_full_STATUS.json"
done_file = OUT_DIR/"task2_full_DONE.txt"
failed_file = OUT_DIR/"task2_full_FAILED.txt"
nohup_log = PROJECT_ROOT/"reproducibility"/"raw_logs"/"task2_full_nohup.log"

if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, 0)
        alive = True
    except OSError:
        alive = False
    print("PID:", pid, "| alive:", alive)
else:
    print("PID file not found.")

if status_file.exists():
    print("\nSTATUS:")
    print(status_file.read_text())

if nohup_log.exists():
    print("\nLAST LOG LINES:")
    lines = nohup_log.read_text(errors="replace").splitlines()
    print("\n".join(lines[-25:]))

if done_file.exists():
    print("\nDONE:")
    print(done_file.read_text())
if failed_file.exists():
    print("\nFAILED:")
    print(failed_file.read_text())


## 8. Tomorrow: one-cell result check

When you return, run only this cell (after setting `PROJECT_ROOT` if needed). It tells you whether everything finished and displays the final metric table.


In [ ]:
from pathlib import Path
import pandas as pd, json

root = PROJECT_ROOT/"task2_sentiment"/"Pramod_Dindukurthi"
out = root/"outputs"

done = out/"task2_full_DONE.txt"
failed = out/"task2_full_FAILED.txt"
status = out/"task2_full_STATUS.json"
metrics = root/"metrics_report.csv"
summary = out/"task2_full_experiment_summary.json"
slice_csv = out/"task2_full_slice_metrics.csv"

if done.exists():
    print("✅ FULL RUN COMPLETED")
    print(done.read_text())
    if metrics.exists():
        df = pd.read_csv(metrics)
        display(df)
        print("\nBest macro-F1:")
        print(df.loc[df["f1_macro"].idxmax(), ["model","accuracy","f1_macro","roc_auc","pr_auc","mcc","brier_score","ece"]])
    if slice_csv.exists():
        print("\nShared slice results:")
        display(pd.read_csv(slice_csv))
    if summary.exists():
        s=json.loads(summary.read_text())
        print("\nMcNemar tests:")
        print(json.dumps(s["mcnemar"], indent=2))
elif failed.exists():
    print("❌ FULL RUN FAILED")
    print(failed.read_text())
else:
    print("⏳ RUN NOT FINISHED")
    if status.exists():
        print(status.read_text())


## 9. Output files you should have after the full run

```text
task2_sentiment/
└── Pramod_Dindukurthi/
    ├── src/
    │   └── task2_runner.py
    ├── data_processed/
    ├── checkpoints/
    │   ├── textcnn/
    │   ├── dpcnn/
    │   └── transformer/
    ├── outputs/
    │   ├── predictions/
    │   ├── plots/
    │   ├── error_analysis/
    │   ├── task2_full_eda_summary.csv
    │   ├── task2_full_slice_metrics.csv
    │   ├── task2_full_metrics_report.csv
    │   ├── task2_full_experiment_summary.json
    │   ├── task2_full_STATUS.json
    │   └── task2_full_DONE.txt
    ├── metrics_report.csv
    ├── results.md
    └── failure_analysis.md

reproducibility/
├── manifests/
│   └── task2_full_hardware.json
└── raw_logs/
    ├── task2_full_master.log
    └── task2_full_nohup.log
```

### Final rubric coverage automatically produced

Per model:
- Accuracy
- Precision macro/micro/weighted
- Recall macro/micro/weighted
- F1 macro/micro/weighted
- Confusion matrix
- ROC-AUC
- PR-AUC
- MCC
- Brier score
- Expected Calibration Error
- 95% bootstrap CI for accuracy / macro-F1 / MCC
- Parameter count
- Training time
- examples/sec
- Peak GPU memory
- Gradient norm / NaN count

Comparative:
- Paired McNemar: TextCNN vs DPCNN
- Paired McNemar: TextCNN vs Transformer
- Macro-F1 and error rate for:
  - short
  - medium
  - long
  - has_negation
  - has_contrast

Error analysis:
- 5 confident false positives
- 5 confident false negatives
- 5 near-threshold errors
- 5 failures from the worst-performing shared slice

The only part that still requires your manual review is writing the qualitative explanation and one testable fix for each of the 20 selected errors.
